<a href="https://colab.research.google.com/github/Mohanee28/06July2024/blob/main/ATL_(3)%20real%20time%20project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Read your existing CSV file
df = pd.read_csv("ATL.csv")

In [ ]:

df.shape

(155, 23)

In [ ]:
# Calculate Disruption Probability (Y)
df['Disruption_probability'] = (df['arr_del15'] + df['arr_cancelled'] + df['arr_diverted']) / df['arr_flights']

In [ ]:
df.columns

Index(['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name',
       'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct',
       'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted',
       'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay',
       'security_delay', 'late_aircraft_delay', 'Y_disruption_probability',
       'Disruption_probability'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# -------------------------------
# Features & target
X = df[['year', 'month', 'airport', 'arr_flights', 'arr_del15', 'arr_cancelled', 'arr_diverted']]
y = df['Disruption_probability']

# -------------------------------
# Identify categorical and numeric columns
categorical_features = ['year', 'month', 'airport']
numeric_features = ['arr_flights', 'arr_del15', 'arr_cancelled', 'arr_diverted']

# -------------------------------
# Chronological train/test split

X_train = X[X['year'].astype(int) <= 2021]
y_train = y[X['year'].astype(int) <= 2021]

X_test = X[X['year'].astype(int) >= 2022]
y_test = y[X['year'].astype(int) >= 2022]

print("✅ Training samples:", X_train.shape[0])
print("✅ Testing samples:", X_test.shape[0])

# -------------------------------
# Preprocessing pipeline

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# -------------------------------
# SVR model

baseline_svr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('regressor', SVR())])

# Train
baseline_svr_pipeline.fit(X_train, y_train)

# Predict
y_pred_baseline = baseline_svr_pipeline.predict(X_test)

# Evaluate
print("\n📊 SVR ")
print("R² Score:", round(r2_score(y_test, y_pred_baseline), 4))
print("MSE:", round(mean_squared_error(y_test, y_pred_baseline), 4))
print("MAE:", round(mean_absolute_error(y_test, y_pred_baseline), 4))

✅ Training samples: 119
✅ Testing samples: 36

📊 SVR 
R² Score: 0.5322
MSE: 0.002
MAE: 0.0403


In [ ]:
# Hyperparameter tuning for SVR


param_grid = {
    'regressor__kernel': ['rbf'],
    'regressor__C': [0.1, 1, 10, 100],
    'regressor__epsilon': [0.01, 0.1, 0.5, 1],
    'regressor__gamma': ['scale', 'auto']
}

svr_search = RandomizedSearchCV(
    baseline_svr_pipeline, param_grid,
    n_iter=10, cv=3, scoring='r2', random_state=42, n_jobs=-1
)

svr_search.fit(X_train, y_train)

# Predict with best model
y_pred_tuned = svr_search.predict(X_test)

print("\n📊 Tuned SVR ")
print("R² Score:", round(r2_score(y_test, y_pred_tuned), 4))
print("MSE:", round(mean_squared_error(y_test, y_pred_tuned), 4))
print("MAE:", round(mean_absolute_error(y_test, y_pred_tuned), 4))



📊 Tuned SVR 
R² Score: 0.8898
MSE: 0.0005
MAE: 0.0158


In [ ]:
# Include year and month from X_test
pred_df = pd.DataFrame({
    'Year': X_test['year'].values,
    'Month': X_test['month'].values,
    'Actual': y_test.values,
    'Svr_Predicted': y_pred_baseline,
    'Tuned_Predicted': y_pred_tuned
})

print("\nSample Predictions \n", pred_df.head(10))



Sample Predictions 
    Year  Month    Actual  Svr_Predicted  Tuned_Predicted
0  2024     12  0.201996       0.238395         0.219285
1  2024     11  0.129849       0.183678         0.115137
2  2024     10  0.141279       0.189126         0.128920
3  2024      9  0.134882       0.190339         0.117442
4  2024      8  0.199153       0.227264         0.197362
5  2024      7  0.346701       0.275813         0.289871
6  2024      6  0.215306       0.234921         0.225229
7  2024      5  0.269196       0.269387         0.258974
8  2024      4  0.171349       0.228429         0.165837
9  2024      3  0.192741       0.237957         0.183729


In [ ]:
# =====================================================
# 7. 2025 JAN–JUL
# =====================================================

future_df = pd.DataFrame({
    'year': [2025]*7,
    'month': [1, 2, 3, 4, 5, 6, 7]
})

# =====================================================
# 8. HISTORICAL AVERAGES (AIRPORT + MONTH)
# =====================================================

historical_avg = (
    df.groupby(['airport', 'month'])[
        ['arr_flights', 'arr_del15',
         'arr_cancelled', 'arr_diverted']
    ]
    .mean()
    .reset_index()
)

# =====================================================
# 9. EXPAND FUTURE DATA FOR ALL AIRPORTS
# =====================================================

airports = df['airport'].unique()

future_forecast_df = future_df.merge(
    pd.DataFrame({'airport': airports}),
    how='cross'
)

future_forecast_df = future_forecast_df.merge(
    historical_avg,
    on=['airport', 'month'],
    how='left'
)

print("\n✅ Future forecast shape:", future_forecast_df.shape)

# =====================================================
# 10. FORECAST DISRUPTION PROBABILITY (2025)
# =====================================================

future_forecast_df['Predicted_Disruption_Probability'] = (
    svr_search.best_estimator_.predict(future_forecast_df)
)

# =====================================================
# 11. FINAL FORECAST OUTPUT
# =====================================================

forecast_output = future_forecast_df[[
    'year', 'month', 'airport',
    'Predicted_Disruption_Probability'
]].sort_values(['airport', 'month'])

forecast_output


✅ Future forecast shape: (7, 7)


,year,month,airport,Predicted_Disruption_Probability
0,2025,1,ATL,0.203438
1,2025,2,ATL,0.163175
2,2025,3,ATL,0.170857
3,2025,4,ATL,0.213382
4,2025,5,ATL,0.197988
5,2025,6,ATL,0.259701
6,2025,7,ATL,0.250308
